<a href="https://colab.research.google.com/github/JulioLDS/Compara-o-de-Desempenho-WEKA-x-Python-/blob/main/Limpezadatasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Pré Processamento


In [ ]:
pip install liac-arff

In [ ]:
import pandas as pd
import numpy as np

def dataframe_to_arff(df, relation_name, filename):
    """Converte um DataFrame Pandas para o formato ARFF do WEKA nativamente."""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"@relation {relation_name}\n\n")

        for col in df.columns:
            # Trata nomes de colunas com caracteres especiais ou espaços
            col_name = f"'{col}'" if any(c in col for c in [' ', '-', '>', '<', '?']) else col

            # Identifica se a coluna é numérica ou nominal/categórica
            if pd.api.types.is_numeric_dtype(df[col]):
                f.write(f"@attribute {col_name} numeric\n")
            else:
                unique_vals = df[col].astype(str).unique()
                formatted_vals = [f"'{str(v).replace('\'', '\\\'')}'" for v in unique_vals]
                vals_str = "{" + ", ".join(formatted_vals) + "}"
                f.write(f"@attribute {col_name} {vals_str}\n")

        f.write("\n@data\n")

        for row in df.itertuples(index=False):
            row_vals = []
            for val in row:
                if pd.isna(val):
                    row_vals.append("?")
                elif isinstance(val, (int, float, np.number)):
                    row_vals.append(str(val))
                else:
                    val_clean = str(val).replace("'", "\\'")
                    row_vals.append(f"'{val_clean}'")
            f.write(",".join(row_vals) + "\n")

    print(f"[OK] ARFF gerado com sucesso: {filename}")

# ==========================================
# 1. PROCESSAMENTO: TELCO (PEQUENO)
# ==========================================
print("Processando Telco...")
df_telco = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Remover ID
df_telco = df_telco.drop(columns=['customerID'], errors='ignore')

# Tratar TotalCharges (corrigido sem inplace)
df_telco['TotalCharges'] = pd.to_numeric(df_telco['TotalCharges'].astype(str).str.strip(), errors='coerce')
df_telco['TotalCharges'] = df_telco['TotalCharges'].fillna(df_telco['TotalCharges'].median())

# Exportar CSV limpo e ARFF
df_telco.to_csv('telco_clean.csv', index=False)
dataframe_to_arff(df_telco, 'telco_clean', 'telco_clean.arff')

# ==========================================
# 2. PROCESSAMENTO: HRTEST (MÉDIO)
# ==========================================
print("\nProcessando HRTest...")
df_hr = pd.read_csv('hrtest.csv')  # ou extraído do seu zip

# Remover ID
df_hr = df_hr.drop(columns=['employee_id'], errors='ignore')

# Tratar Nulos (corrigido sem inplace)
df_hr['education'] = df_hr['education'].fillna("Bachelor's")
df_hr['previous_year_rating'] = df_hr['previous_year_rating'].fillna(df_hr['previous_year_rating'].median())

# Exportar CSV limpo e ARFF
df_hr.to_csv('hr_clean.csv', index=False)
dataframe_to_arff(df_hr, 'hr_clean', 'hr_clean.arff')

# ==========================================
# 3. PROCESSAMENTO: MARKETING (COLOSSAL)
# ==========================================
print("\nProcessando Marketing...")
df_mkt = pd.read_csv('marketing_campaign_dataset.csv')

# Preencher nulos numéricos se houver
num_cols = df_mkt.select_dtypes(include=[np.number]).columns
df_mkt[num_cols] = df_mkt[num_cols].fillna(df_mkt[num_cols].median())

# Exportar CSV limpo e ARFF
df_mkt.to_csv('marketing_clean.csv', index=False)
dataframe_to_arff(df_mkt, 'marketing_clean', 'marketing_clean.arff')

print("\n--- PASSO 1 CONCLUÍDO COM SUCESSO! ---")

Processando Telco...
[OK] ARFF gerado com sucesso: telco_clean.arff

Processando HRTest...
[OK] ARFF gerado com sucesso: hr_clean.arff

Processando Marketing...
[OK] ARFF gerado com sucesso: marketing_clean.arff

--- PASSO 1 CONCLUÍDO COM SUCESSO! ---
